# Eurex GraphQL API Tutorial

[![Open In Colab](https://img.shields.io/badge/Google%20Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=white)](https://colab.research.google.com/github/ViktorHD/eurex-api/blob/main/notebooks/eurex_graphql_tutorial.ipynb) 
[![Binder](https://img.shields.io/badge/Binder-579ACA?style=for-the-badge&logo=binder&logoColor=white)](https://mybinder.org/v2/gh/ViktorHD/eurex-api/main?labpath=notebooks%2Feurex_graphql_tutorial.ipynb)

This notebook provides a complete guide for Python users to interact with the [Eurex GraphQL API](https://www.eurex.com/ex-en/data/free-reference-data-api).

## 1. Setup & Authentication

To access the API, you need an API key from the Deutsche Börse Digital Business Platform.

In [ ]:
import requests
import json
import pandas as pd

API_URL = "https://api.developer.deutsche-boerse.com/eurex-prod-graphql/"
API_KEY = "YOUR_API_KEY_HERE" # Replace with your actual key

headers = {
    "Content-Type": "application/json",
    "X-DBP-APIKEY": API_KEY
}

## 2. Basic Querying

The most basic way to query the API is to use the `requests` library to send a POST request with a GraphQL query string.

In [ ]:
def run_query(query):
    response = requests.post(API_URL, json={'query': query}, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Query failed with status code {response.status_code}: {response.text}")

# Simple query for all products
query = """
query {
  Products {
    data {
      Product
      Name
    }
  }
}
"""

result = run_query(query)
products_df = pd.DataFrame(result['data']['Products']['data'])
print(f"Found {len(products_df)} products")
products_df.head()

## 3. Fetching All Contracts (Recommended Pattern)

The Eurex API does not support fetching the entire contract universe in a single query due to its size. The recommended pattern is to first fetch all products, and then loop over them to fetch contracts for each product.

In [ ]:
def fetch_all_contracts():
    # 1. Fetch all products
    products_query = """
    query {
      Products {
        data {
          Product
        }
      }
    }
    """
    products = run_query(products_query)['data']['Products']['data']
    
    all_contracts = []
    
    # 2. Loop over products to fetch contracts
    for p in products:
        product_code = p['Product']
        print(f"Fetching contracts for {product_code}...")
        
        contract_query = f"""
        query {{
          Contracts(filter: {{ Product: {{ eq: \"{product_code}\" }} }}) {{
            data {{
              Product
              Contract
              ISIN
              ExpirationDate
            }}
          }}
        }}
        """
        try:
            contracts = run_query(contract_query)['data']['Contracts']['data']
            all_contracts.extend(contracts)
        except Exception as e:
            print(f"Error fetching {product_code}: {e}")
            
    return pd.DataFrame(all_contracts)

# Warning: This might take a while depending on the number of products
# contracts_df = fetch_all_contracts()
# print(f"Total contracts fetched: {len(contracts_df)}")